# BARAM 2026 Preprocessing Reference

This notebook is a preprocessing-only reference for the wind generation task.

It builds reusable group-specific feature frames and stops before modeling:

- `group_feature_sets[target]["features"]`
- `group_feature_sets[target]["train_df"]`
- `group_feature_sets[target]["test_df"]`
- `group_feature_sets[target]["feature_cols"]`

Use this notebook as the base feature engineering recipe for tree models, deep learning models, or later ablation experiments.

## 1. Imports and Configuration

Set runtime paths, output folders, target names, capacity values, feature windows, and group-specific LDAPS grid plans.

In [ ]:
from pathlib import Path
import gc
import json
import random
import time

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

COLAB_DATA_DIR = Path("/content/drive/MyDrive/BARAM2026/data")
LOCAL_DATA_DIR = Path(r"C:\Users\심현석\Documents\BARAM 2026")

if IN_COLAB:
    BASE_DIR = COLAB_DATA_DIR
elif LOCAL_DATA_DIR.exists():
    BASE_DIR = LOCAL_DATA_DIR
else:
    BASE_DIR = Path.cwd()

TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAME = "preprocess_reference"
CHECKPOINT_DIR = OUTPUT_DIR / f"{EXPERIMENT_NAME}_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {
    "kpx_group_1": 21_600.0,
    "kpx_group_2": 21_600.0,
    "kpx_group_3": 21_000.0,
}

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

WEATHER_AGGS = ["mean", "std", "min", "max"]
SUBSET_AGGS = ["mean", "std", "min", "max"]

# Requested lag/rolling settings.
WEATHER_LAGS = [1, 2, 3, 24]
WEATHER_ROLL_WINDOWS = [3, 6, 12, 24]

# Keep this finite by default so the reference notebook can run on a normal Colab/PC.
# Set to None if you intentionally want lag/rolling features for every eligible base column.
MAX_LAG_ROLL_BASE_COLS = 180

FEATURE_INVENTORY_PATH = CHECKPOINT_DIR / "feature_inventory_by_target.csv"

print("BASE_DIR:", BASE_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

## 2. Load Data and Interpolate Internal Label Gaps

Load label, forecast, and submission files. Internal label gaps are filled by time interpolation while group 3's pre-operation 2022 period remains missing.

In [ ]:
required_paths = [
    TRAIN_DIR / "train_labels.csv",
    TRAIN_DIR / "ldaps_train.csv",
    TRAIN_DIR / "gfs_train.csv",
    TEST_DIR / "ldaps_test.csv",
    TEST_DIR / "gfs_test.csv",
    BASE_DIR / "sample_submission.csv",
]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing_paths))

labels_raw = pd.read_csv(TRAIN_DIR / "train_labels.csv")
labels_raw["kst_dtm"] = pd.to_datetime(labels_raw["kst_dtm"])
labels_raw = labels_raw.sort_values("kst_dtm").reset_index(drop=True)


def interpolate_target_labels(labels: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = labels.sort_values("kst_dtm").set_index("kst_dtm").copy()
    audit_rows = []
    for target in TARGET_COLS:
        original = out[target].copy()
        first_valid = original.first_valid_index()
        last_valid = original.last_valid_index()
        if first_valid is None or last_valid is None:
            continue
        within = (original.index >= first_valid) & (original.index <= last_valid)
        before = int(original.loc[within].isna().sum())
        interpolated = original.copy()
        interpolated.loc[within] = original.loc[within].interpolate(method="time", limit_area="inside")
        after = int(interpolated.loc[within].isna().sum())
        out[target] = interpolated
        audit_rows.append({
            "target": target,
            "first_valid_time": first_valid,
            "last_valid_time": last_valid,
            "missing_total_before": int(original.isna().sum()),
            "missing_total_after": int(interpolated.isna().sum()),
            "missing_internal_before": before,
            "missing_internal_after": after,
            "filled_internal_count": before - after,
            "note": "Group 3 2022 remains NaN before first_valid_time." if target == "kpx_group_3" else "",
        })
    return out.reset_index(), pd.DataFrame(audit_rows)


labels, label_interpolation_audit = interpolate_target_labels(labels_raw)

ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv")
sample_submission = pd.read_csv(BASE_DIR / "sample_submission.csv")

for df in [ldaps_train, gfs_train, ldaps_test, gfs_test]:
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    df["data_available_kst_dtm"] = pd.to_datetime(df["data_available_kst_dtm"])

sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

print("labels:", labels.shape, labels["kst_dtm"].min(), labels["kst_dtm"].max())
print("ldaps_train/test:", ldaps_train.shape, ldaps_test.shape)
print("gfs_train/test:", gfs_train.shape, gfs_test.shape)
print("sample_submission:", sample_submission.shape)
display(label_interpolation_audit)

## 3. Drop Excluded Weather Variables

Remove selected weather columns before feature generation so the downstream feature set stays focused on wind-related signals.

In [ ]:
LDAPS_DROP_COLS = [
    "etc_0_blh",
    "surface_0_NDNSW",
    "surface_0_NDNLW",
    "heightAboveGround_2_SWDIR",
    "heightAboveGround_2_SWDIF",
    "etc_0_hcc",
    "etc_0_mcc",
    "etc_0_lcc",
    "etc_0_VLCDC",
    "surface_0_avg_lsprate",
    "surface_0_lssrate",
    "surface_0_ncpcp",
    "surface_0_snol",
    "surface_0_SNOM",
    "surface_0_lsm",
    "surface_0_h",
]

GFS_DROP_COLS = [
    "planetaryBoundaryLayer_0_u",
    "planetaryBoundaryLayer_0_v",
    "planetaryBoundaryLayer_0_VRATE",
    "surface_0_dswrf",
    "surface_0_dlwrf",
    "surface_0_prate",
    "surface_0_tp",
    "lowCloudLayer_0_lcc",
    "middleCloudLayer_0_mcc",
    "highCloudLayer_0_hcc",
    "atmosphere_0_tcc",
]


def drop_existing_columns(df: pd.DataFrame, cols: list[str], name: str) -> pd.DataFrame:
    existing = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if existing:
        df = df.drop(columns=existing)
    print(
        f"{name}: dropped {len(existing)} columns; "
        f"remaining columns={df.shape[1]}; missing_from_drop_list={len(missing)}"
    )
    if missing:
        print(f"{name} missing columns:", missing)
    return df


ldaps_train = drop_existing_columns(ldaps_train, LDAPS_DROP_COLS, "ldaps_train")
ldaps_test = drop_existing_columns(ldaps_test, LDAPS_DROP_COLS, "ldaps_test")
gfs_train = drop_existing_columns(gfs_train, GFS_DROP_COLS, "gfs_train")
gfs_test = drop_existing_columns(gfs_test, GFS_DROP_COLS, "gfs_test")

## 4. Feature Engineering Helpers

Define the prediction base time, wind vector transformations, air-density features, inverse wind-speed features, and group-specific wind regime thresholds.

In [ ]:
def prediction_base_time(ts: pd.Series | pd.DatetimeIndex) -> pd.Series | pd.DatetimeIndex:
    dt = pd.to_datetime(ts)
    if isinstance(dt, pd.Series):
        forecast_day = dt.dt.normalize()
        forecast_day = forecast_day - pd.to_timedelta((dt.dt.hour == 0).astype(int), unit="D")
        return forecast_day - pd.Timedelta(days=1) + pd.Timedelta(hours=13)
    idx = pd.DatetimeIndex(dt)
    forecast_day = idx.normalize() - pd.to_timedelta((idx.hour == 0).astype(int), unit="D")
    return forecast_day - pd.Timedelta(days=1) + pd.Timedelta(hours=13)


LDAPS_UV_PAIRS = [
    ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "wind_10m"),
    ("heightAboveGround_50_50MUmax", "heightAboveGround_50_50MVmax", "wind_50m_max"),
    ("heightAboveGround_50_50MUmin", "heightAboveGround_50_50MVmin", "wind_50m_min"),
    ("heightAboveGround_5_XBLWS", "heightAboveGround_5_YBLWS", "xbl_wind_5m"),
]

GFS_UV_PAIRS = [
    ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "wind_10m"),
    ("heightAboveGround_80_u", "heightAboveGround_80_v", "wind_80m"),
    ("heightAboveGround_100_100u", "heightAboveGround_100_100v", "wind_100m"),
    ("planetaryBoundaryLayer_0_u", "planetaryBoundaryLayer_0_v", "pbl_wind"),
    ("isobaricInhPa_850_u", "isobaricInhPa_850_v", "wind_850hpa"),
    ("isobaricInhPa_700_u", "isobaricInhPa_700_v", "wind_700hpa"),
    ("isobaricInhPa_500_u", "isobaricInhPa_500_v", "wind_500hpa"),
]

LDAPS_SPEED_COLS = [f"{name}_speed" for _, _, name in LDAPS_UV_PAIRS]
GFS_SPEED_COLS = [f"{name}_speed" for _, _, name in GFS_UV_PAIRS]

GROUP_REGIME_SPECS = {
    "kpx_group_1": {"alias": "g1", "cut_in": 3.0, "rated": 12.0, "cut_out": 22.5},
    "kpx_group_2": {"alias": "g2", "cut_in": 3.0, "rated": 12.0, "cut_out": 22.5},
    "kpx_group_3": {"alias": "g3", "cut_in": 3.0, "rated": 12.5, "cut_out": 22.0},
}

LDAPS_GROUP_SPATIAL_PLAN = {
    "kpx_group_1": {
        "raw_grid_ids": [5, 6, 10],
        "subsets": {
            "g1_s_5_6_10_11": [5, 6, 10, 11],
            "g1_s_5_6_1_2": [5, 6, 1, 2],
        },
    },
    "kpx_group_2": {
        "raw_grid_ids": [6, 11],
        "subsets": {
            "g2_s_6_5_7_11": [6, 5, 7, 11],
            "g2_s_6_7_11_12": [6, 7, 11, 12],
            "g2_s_6_11_12_15": [6, 11, 12, 15],
        },
    },
    "kpx_group_3": {
        "raw_grid_ids": [6, 12],
        "subsets": {
            "g3_s_6_7_11_12": [6, 7, 11, 12],
            "g3_s_7_11_12_16": [7, 11, 12, 16],
            "g3_s_11_12_15_16": [11, 12, 15, 16],
        },
    },
}

GFS_RAW_GRID_IDS = [5]


def safe_inverse(values: pd.Series, power: int = 1) -> pd.Series:
    vals = values.astype(float)
    return pd.Series(np.where(vals > 0, 1.0 / np.power(vals, power), np.nan), index=values.index)


def add_air_density_features(out: pd.DataFrame) -> pd.DataFrame:
    temp_candidates = [c for c in ["heightAboveGround_2_t", "heightAboveGround_2_2t"] if c in out.columns]
    q_candidates = [c for c in ["heightAboveGround_2_q", "heightAboveGround_2_2sh"] if c in out.columns]
    pressure_candidates = [c for c in ["surface_0_sp"] if c in out.columns]
    if not temp_candidates or not pressure_candidates:
        return out

    temp_k = out[temp_candidates[0]].astype(float).clip(lower=150.0, upper=360.0)
    pressure_pa = out[pressure_candidates[0]].astype(float).clip(lower=50_000.0, upper=110_000.0)
    out["dry_air_density"] = pressure_pa / (287.05 * temp_k)
    out["dry_air_density_inv"] = safe_inverse(out["dry_air_density"], power=1)

    if q_candidates:
        specific_humidity = out[q_candidates[0]].astype(float).clip(lower=0.0, upper=0.08)
        out["humid_air_density"] = pressure_pa / (287.05 * temp_k * (1.0 + 0.61 * specific_humidity))
        out["humid_air_density_inv"] = safe_inverse(out["humid_air_density"], power=1)

    # Deliberately do not create `air_density`, because it duplicates humid_air_density.
    return out


def add_wind_physics_features(df: pd.DataFrame, uv_pairs: list[tuple[str, str, str]]) -> pd.DataFrame:
    out = df.copy()
    speed_cols = []
    for u_col, v_col, name in uv_pairs:
        if u_col not in out.columns or v_col not in out.columns:
            continue

        u = out[u_col].astype(float)
        v = out[v_col].astype(float)
        speed = np.sqrt(u * u + v * v)
        angle = np.arctan2(v, u)
        speed_col = f"{name}_speed"

        out[speed_col] = speed
        out[f"{name}_speed_sq"] = speed ** 2
        out[f"{name}_speed_cubed"] = speed ** 3
        out[f"{name}_speed_inv_sq"] = safe_inverse(speed, power=2)
        out[f"{name}_speed_inv_cubed"] = safe_inverse(speed, power=3)
        out[f"{name}_angle_sin"] = np.sin(angle)
        out[f"{name}_angle_cos"] = np.cos(angle)
        speed_cols.append(speed_col)

    out = add_air_density_features(out)

    density_col = "humid_air_density" if "humid_air_density" in out.columns else "dry_air_density"
    if density_col in out.columns:
        for speed_col in speed_cols:
            out[f"{speed_col}_power_density"] = 0.5 * out[density_col] * out[speed_col].astype(float).clip(lower=0) ** 3
    return out

## 5. Weather Grid Preparation and Spatial Aggregation Helpers

Filter forecasts by data availability, keep the latest safe forecast per grid and target time, then reshape or aggregate grid-level weather features.

In [ ]:
def prepare_weather_grid(df: pd.DataFrame, prefix: str, uv_pairs: list[tuple[str, str, str]]) -> pd.DataFrame:
    out = df.copy()
    out["forecast_kst_dtm"] = pd.to_datetime(out["forecast_kst_dtm"])
    out["data_available_kst_dtm"] = pd.to_datetime(out["data_available_kst_dtm"])
    out = add_wind_physics_features(out, uv_pairs)

    out["prediction_base_kst_dtm"] = prediction_base_time(out["forecast_kst_dtm"])
    safe_mask = out["data_available_kst_dtm"] <= out["prediction_base_kst_dtm"]
    unsafe_count = int((~safe_mask).sum())
    if unsafe_count:
        print(f"[WARN] {prefix}: dropping {unsafe_count:,} unsafe forecast rows.")
    out = out.loc[safe_mask].copy()
    out = out.sort_values(["forecast_kst_dtm", "grid_id", "data_available_kst_dtm"])
    out = out.groupby(["forecast_kst_dtm", "grid_id"], as_index=False).tail(1)
    out[f"{prefix}_lead_hours"] = (out["forecast_kst_dtm"] - out["data_available_kst_dtm"]).dt.total_seconds() / 3600
    out[f"{prefix}_available_before_base_hours"] = (
        out["prediction_base_kst_dtm"] - out["data_available_kst_dtm"]
    ).dt.total_seconds() / 3600
    return out.drop(columns=["prediction_base_kst_dtm"])


def add_group_speed_regime_flags(
    grid_df: pd.DataFrame,
    target: str,
    speed_cols: list[str],
) -> pd.DataFrame:
    spec = GROUP_REGIME_SPECS[target]
    out = grid_df.copy()
    cut_in = spec["cut_in"]
    rated = spec["rated"]
    cut_out = spec["cut_out"]
    for speed_col in speed_cols:
        if speed_col not in out.columns:
            continue
        speed = out[speed_col].astype(float)
        out[f"{speed_col}_below_cutin"] = (speed < cut_in).astype(float)
        out[f"{speed_col}_cutin_to_rated"] = ((speed >= cut_in) & (speed < rated)).astype(float)
        out[f"{speed_col}_rated_to_cutout"] = ((speed >= rated) & (speed < cut_out)).astype(float)
    return out


def regime_flag_cols(df: pd.DataFrame) -> list[str]:
    suffixes = ("_below_cutin", "_cutin_to_rated", "_rated_to_cutout")
    return [c for c in df.columns if c.endswith(suffixes)]


def numeric_weather_cols(df: pd.DataFrame, exclude_regime_flags: bool = False) -> list[str]:
    skip = {"forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    cols = [c for c in df.columns if c not in skip and pd.api.types.is_numeric_dtype(df[c])]
    if exclude_regime_flags:
        flag_set = set(regime_flag_cols(df))
        cols = [c for c in cols if c not in flag_set]
    return cols


def aggregate_all_grids(grid_df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = numeric_weather_cols(grid_df, exclude_regime_flags=True)
    agg = grid_df.groupby("forecast_kst_dtm", sort=True)[cols].agg(WEATHER_AGGS)
    agg.columns = [f"{prefix}_all_{col}_{stat}" for col, stat in agg.columns]
    return agg.reset_index()


def raw_grid_wide(grid_df: pd.DataFrame, prefix: str, grid_ids: list[int]) -> pd.DataFrame:
    cols = numeric_weather_cols(grid_df, exclude_regime_flags=False)
    frames = []
    for gid in grid_ids:
        sub = grid_df.loc[grid_df["grid_id"] == gid, ["forecast_kst_dtm", *cols]].copy()
        rename = {c: f"{prefix}_g{gid:02d}_{c}" for c in cols}
        frames.append(sub.rename(columns=rename))

    if not frames:
        return pd.DataFrame({"forecast_kst_dtm": pd.Series(dtype="datetime64[ns]")})

    out = frames[0]
    for frame in frames[1:]:
        out = out.merge(frame, on="forecast_kst_dtm", how="outer")
    return out


def aggregate_subset(
    grid_df: pd.DataFrame,
    prefix: str,
    subset_name: str,
    grid_ids: list[int],
) -> pd.DataFrame:
    base_cols = numeric_weather_cols(grid_df, exclude_regime_flags=True)
    flag_cols = regime_flag_cols(grid_df)
    sub = grid_df.loc[grid_df["grid_id"].isin(grid_ids), ["forecast_kst_dtm", "grid_id", *base_cols, *flag_cols]].copy()

    pieces = []
    if base_cols:
        agg = sub.groupby("forecast_kst_dtm", sort=True)[base_cols].agg(SUBSET_AGGS)
        agg.columns = [f"{prefix}_{subset_name}_{col}_{stat}" for col, stat in agg.columns]
        pieces.append(agg.reset_index())

    if flag_cols:
        counts = sub.groupby("forecast_kst_dtm", sort=True)[flag_cols].sum(numeric_only=True).reset_index()
        counts = counts.rename(columns={c: f"{prefix}_{subset_name}_{c}_count" for c in flag_cols})
        pieces.append(counts)

    if not pieces:
        return pd.DataFrame({"forecast_kst_dtm": pd.Series(dtype="datetime64[ns]")})

    out = pieces[0]
    for frame in pieces[1:]:
        out = out.merge(frame, on="forecast_kst_dtm", how="outer")
    return out

## 6. Calendar, Lag and Rolling Helpers

Create calendar encodings and past-only lag/rolling features. Day-of-year cyclic encoding uses a non-leap 365-day calendar alignment.

In [ ]:
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out["forecast_kst_dtm"])
    out["hour"] = dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["month"] = dt.dt.month
    out["dayofyear"] = dt.dt.dayofyear
    out["season"] = ((dt.dt.month % 12) // 3).astype(int)
    out["season_sin"] = np.sin(2 * np.pi * out["season"] / 4)
    out["season_cos"] = np.cos(2 * np.pi * out["season"] / 4)
    out["is_weekend"] = (out["dayofweek"] >= 5).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    cycle_day = out["dayofyear"] - ((dt.dt.is_leap_year) & (dt.dt.month > 2)).astype(int)
    out["dayofyear_sin"] = np.sin(2 * np.pi * cycle_day / 365)
    out["dayofyear_cos"] = np.cos(2 * np.pi * cycle_day / 365)
    return out


def choose_lag_roll_cols(df: pd.DataFrame, subset_names: list[str]) -> list[str]:
    keywords = [
        "wind", "speed", "gust", "power_density",
        "dry_air_density", "humid_air_density",
        "below_cutin", "cutin_to_rated", "rated_to_cutout",
        "surface_0_sp",
        "heightAboveGround_2_t", "heightAboveGround_2_2t",
        "heightAboveGround_2_q", "heightAboveGround_2_2sh",
        "_g05_", "_g06_", "_g10_", "_g11_", "_g12_", "_g16_",
    ]
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    candidates = [
        c for c in numeric_cols
        if any(k in c for k in keywords)
        and "_lag" not in c
        and "_roll" not in c
    ]
    if MAX_LAG_ROLL_BASE_COLS is None or len(candidates) <= MAX_LAG_ROLL_BASE_COLS:
        return candidates

    subset_cols = [c for c in candidates if any(subset_name in c for subset_name in subset_names)]
    protected_keywords = [
        "dry_air_density", "humid_air_density",
        "below_cutin", "cutin_to_rated", "rated_to_cutout",
        "_g05_", "_g06_", "_g10_", "_g11_", "_g12_", "_g16_",
    ]
    protected = [c for c in candidates if any(k in c for k in protected_keywords)]
    protected = list(dict.fromkeys(subset_cols + protected))

    remaining_slots = max(MAX_LAG_ROLL_BASE_COLS - len(protected), 0)
    variances = df[candidates].var(numeric_only=True).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    ranked = [c for c in variances.sort_values(ascending=False).index.tolist() if c not in protected]
    selected = (protected + ranked[:remaining_slots])[:MAX_LAG_ROLL_BASE_COLS]
    return selected


def add_lag_rolling_features(df: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    out = df.sort_values("forecast_kst_dtm").reset_index(drop=True).copy()
    new_features = {}
    for col in base_cols:
        if col not in out.columns:
            continue
        s = out[col].astype(float)
        for lag in WEATHER_LAGS:
            new_features[f"{col}_lag{lag}"] = s.shift(lag)
        shifted = s.shift(1)
        for window in WEATHER_ROLL_WINDOWS:
            new_features[f"{col}_roll{window}_mean"] = shifted.rolling(window, min_periods=1).mean()
            new_features[f"{col}_roll{window}_std"] = shifted.rolling(window, min_periods=2).std()
    if new_features:
        out = pd.concat([out, pd.DataFrame(new_features, index=out.index)], axis=1)
    return out


def merge_feature_frames(frames: list[pd.DataFrame]) -> pd.DataFrame:
    non_empty = [f for f in frames if f is not None and len(f.columns) > 1]
    if not non_empty:
        raise ValueError("No feature frames to merge.")
    out = non_empty[0]
    for frame in non_empty[1:]:
        out = out.merge(frame, on="forecast_kst_dtm", how="outer")
    return out


## 7. Build Group-Specific Feature Frames

Combine common LDAPS/GFS features with each group's raw LDAPS grid features and subset aggregation features, then save feature inventories.

In [ ]:
started = time.time()

ldaps_all = pd.concat([ldaps_train, ldaps_test], ignore_index=True)
gfs_all = pd.concat([gfs_train, gfs_test], ignore_index=True)

print("Preparing LDAPS grid-level data...")
ldaps_grid = prepare_weather_grid(ldaps_all, "ldaps", LDAPS_UV_PAIRS)
print("Preparing GFS grid-level data...")
gfs_grid = prepare_weather_grid(gfs_all, "gfs", GFS_UV_PAIRS)

ldaps_locations = ldaps_grid[["grid_id", "latitude", "longitude"]].drop_duplicates().sort_values("grid_id").reset_index(drop=True)
gfs_locations = gfs_grid[["grid_id", "latitude", "longitude"]].drop_duplicates().sort_values("grid_id").reset_index(drop=True)
display(ldaps_locations)
display(gfs_locations)

print("Building common global features...")
ldaps_all_agg = aggregate_all_grids(ldaps_grid, "ldaps")
gfs_all_agg = aggregate_all_grids(gfs_grid, "gfs")
gfs_grid5_raw = raw_grid_wide(gfs_grid, "gfs", GFS_RAW_GRID_IDS)

common_features = merge_feature_frames([ldaps_all_agg, gfs_all_agg, gfs_grid5_raw])
common_features = add_calendar_features(common_features)
common_features = common_features.sort_values("forecast_kst_dtm").reset_index(drop=True)

group_feature_sets = {}
feature_inventory_rows = []

for target in TARGET_COLS:
    print(f"\nBuilding group-specific features for {target}")
    plan = LDAPS_GROUP_SPATIAL_PLAN[target]
    subset_names = list(plan["subsets"].keys())

    ldaps_group_grid = add_group_speed_regime_flags(ldaps_grid, target, LDAPS_SPEED_COLS)
    group_frames = [common_features]

    raw_ids = plan["raw_grid_ids"]
    group_frames.append(raw_grid_wide(ldaps_group_grid, "ldaps", raw_ids))

    for subset_name, grid_ids in plan["subsets"].items():
        print(target, subset_name, "basic aggs + regime counts:", grid_ids)
        group_frames.append(aggregate_subset(ldaps_group_grid, "ldaps", subset_name, grid_ids))

    features = merge_feature_frames(group_frames)
    features = features.sort_values("forecast_kst_dtm").reset_index(drop=True)

    lag_base_cols = choose_lag_roll_cols(features, subset_names)
    print(target, "lag/rolling base cols:", len(lag_base_cols))
    features = add_lag_rolling_features(features, lag_base_cols)

    train_df = labels.rename(columns={"kst_dtm": "forecast_kst_dtm"}).merge(features, on="forecast_kst_dtm", how="left")
    test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(features, on="forecast_kst_dtm", how="left")

    if len(test_df) != len(sample_submission):
        numeric_cols = [
            c for c in test_df.columns
            if c not in ["forecast_id", "forecast_kst_dtm"] and pd.api.types.is_numeric_dtype(test_df[c])
        ]
        collapsed = test_df.groupby(["forecast_id", "forecast_kst_dtm"], as_index=False)[numeric_cols].mean(numeric_only=True)
        test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(
            collapsed,
            on=["forecast_id", "forecast_kst_dtm"],
            how="left",
        )
    assert len(test_df) == len(sample_submission)

    non_feature_cols = {"forecast_id", "forecast_kst_dtm", *TARGET_COLS}
    datetime_cols = [c for c in train_df.columns if str(train_df[c].dtype).startswith("datetime")]
    non_feature_cols.update(datetime_cols)
    feature_cols = [
        c for c in train_df.columns
        if c not in non_feature_cols and pd.api.types.is_numeric_dtype(train_df[c])
    ]

    combined = pd.concat([train_df[feature_cols], test_df[feature_cols]], ignore_index=True)
    keep_cols = [c for c in feature_cols if combined[c].notna().sum() > 0 and combined[c].nunique(dropna=True) > 1]
    del combined
    gc.collect()

    group_feature_sets[target] = {
        "features": features,
        "train_df": train_df,
        "test_df": test_df,
        "feature_cols": keep_cols,
        "lag_base_cols": lag_base_cols,
    }

    pd.DataFrame({"feature": keep_cols}).to_csv(
        CHECKPOINT_DIR / f"{target}_feature_columns.csv",
        index=False,
        encoding="utf-8-sig",
    )
    pd.DataFrame({"feature": lag_base_cols}).to_csv(
        CHECKPOINT_DIR / f"{target}_lag_rolling_base_columns.csv",
        index=False,
        encoding="utf-8-sig",
    )
    feature_inventory_rows.append({
        "target": target,
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "feature_count": len(keep_cols),
        "lag_base_count": len(lag_base_cols),
        "raw_ldaps_grid_ids": json.dumps(raw_ids),
        "subset_names": json.dumps(subset_names),
        "weather_lags": json.dumps(WEATHER_LAGS),
        "weather_roll_windows": json.dumps(WEATHER_ROLL_WINDOWS),
    })
    print(target, "train/test/features:", train_df.shape, test_df.shape, len(keep_cols))

feature_inventory = pd.DataFrame(feature_inventory_rows)
feature_inventory.to_csv(FEATURE_INVENTORY_PATH, index=False, encoding="utf-8-sig")
display(feature_inventory)
print("Feature build elapsed_sec:", round(time.time() - started, 1))
print("Saved feature inventory:", FEATURE_INVENTORY_PATH)

## 8. Feature Output Preview

Print a compact summary of the generated feature frames and show representative feature names by group and category.

In [ ]:
def classify_feature_name(feature: str) -> str:
    if "_lag" in feature:
        return "lag"
    if "_roll" in feature:
        return "rolling"
    if "below_cutin" in feature or "cutin_to_rated" in feature or "rated_to_cutout" in feature:
        return "wind_regime"
    if "speed_inv_sq" in feature or "speed_inv_cubed" in feature:
        return "inverse_wind_speed"
    if "power_density" in feature:
        return "wind_power_density"
    if "dry_air_density" in feature or "humid_air_density" in feature:
        return "air_density"
    if feature.startswith("ldaps_g"):
        return "ldaps_raw_grid"
    if feature.startswith("ldaps_all_"):
        return "ldaps_all_grid_agg"
    if feature.startswith("ldaps_"):
        return "ldaps_subset_agg"
    if feature.startswith("gfs_g"):
        return "gfs_raw_grid"
    if feature.startswith("gfs_all_"):
        return "gfs_all_grid_agg"
    if feature in {
        "hour", "dayofweek", "month", "dayofyear", "season", "season_sin", "season_cos",
        "is_weekend", "hour_sin", "hour_cos", "month_sin", "month_cos",
        "dayofyear_sin", "dayofyear_cos",
    }:
        return "calendar"
    return "other"


summary_rows = []
category_frames = []

for target, fs in group_feature_sets.items():
    feature_cols = fs["feature_cols"]
    train_df = fs["train_df"]
    test_df = fs["test_df"]

    summary_rows.append({
        "target": target,
        "train_shape": str(train_df.shape),
        "test_shape": str(test_df.shape),
        "feature_count": len(feature_cols),
        "lag_base_count": len(fs["lag_base_cols"]),
        "saved_feature_list": str(CHECKPOINT_DIR / f"{target}_feature_columns.csv"),
    })

    categories = pd.Series([classify_feature_name(c) for c in feature_cols], name="category")
    cat_count = categories.value_counts().rename_axis("category").reset_index(name="n_features")
    cat_count.insert(0, "target", target)
    category_frames.append(cat_count)

summary_df = pd.DataFrame(summary_rows)
category_summary = pd.concat(category_frames, ignore_index=True)

display(summary_df)
display(category_summary.sort_values(["target", "n_features"], ascending=[True, False]))

for target, fs in group_feature_sets.items():
    print(f"\n[{target}] first 80 generated feature columns")
    display(pd.DataFrame({"feature": fs["feature_cols"][:80]}))

print("Feature inventory CSV:", FEATURE_INVENTORY_PATH)
print("Checkpoint directory:", CHECKPOINT_DIR)